# External validation, every cohort alike

Redraws the external validation with **no lead cohort**: the eight bulk substantia nigra cohorts found for
`pd-lcm-rf-external-multi` are analysed together (GSE7621 is one of the eight, nothing more). This notebook
reads only that notebook's outputs - nothing is downloaded or refitted.

* **Figure 6** - a: ROC of the frozen models, averaged over the cohorts; b: how much of the signal is neuron loss;
  c: every Boruta gene in every cohort
* **Figure 7** - a: every cohort's AUC; b: calls at cut-offs fixed before testing; c: which brains were shared with
  the discovery data

In [ ]:
DISC_N = {'GSE20141': 18, 'GSE24378': 17, 'GSE182622': 22, 'GSE169755': 6}   # discovery people per study

## 1. Read the multi-cohort outputs and pool them

In [ ]:
import os, glob, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict
warnings.filterwarnings("ignore")
ON_KAGGLE = Path("/kaggle/input").exists()
OUT = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("FIG_OUT", "."))
def find_in(name):
    if not ON_KAGGLE:
        return Path(os.environ["EXT_IN"]) / name
    hits = sorted((h for h in glob.glob(f"/kaggle/input/**/{name}", recursive=True) if "external-multi" in h), key=len)
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]
RES = pd.read_csv(find_in("external_multi_results.csv")).set_index(["cohort", "model"])
SCO = pd.read_csv(find_in("external_multi_scores.csv"))
GBC = pd.read_csv(find_in("external_multi_gene_by_cohort.csv"))
GMT = pd.read_csv(find_in("external_multi_gene_meta.csv"))
OVL = pd.read_csv(find_in("donor_overlap.csv")).set_index("cohort")
MATCH = pd.read_csv(find_in("donor_overlap_matches.csv"))
SUM = json.load(open(find_in("external_multi_summary.json")))
POOL = SUM["pooled"]["all"]                       # every cohort, each donor once - the analysis reported
COHORTS = ["GSE7621", "GSE8397", "GSE20163", "GSE20164", "GSE20292", "GSE49036", "GSE114517", "GSE168496"]   # by accession
assert sorted(COHORTS) == sorted(POOL["cohorts"])
P = SCO[SCO.counted_in_pool].copy()               # shared donors already removed; the one repeated external donor counted once

def fast_auc(yv, s):
    r = rankdata(s); n1 = int(yv.sum()); n0 = len(yv) - n1
    return (r[yv == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
BY = {c: P[P.cohort == c] for c in COHORTS}
W = np.array([d.y.sum() * (1 - d.y).sum() for d in BY.values()], float)       # PD-control pairs per cohort
def pooled(fn):
    return float(np.sum(W * np.array([fn(d) for d in BY.values()])) / W.sum())
SCORE = {"core": lambda d: d.frozen.to_numpy(), "panel": lambda d: d.panel.to_numpy(), "neuron": lambda d: -d.neuron_score.to_numpy()}

# ROC averaged over cohorts: each cohort's curve, averaged vertically with the same pair weights as the pooled AUC
GRID = np.linspace(0, 1, 1001)
def roc_xy(yv, s):
    order = np.argsort(-s, kind="mergesort"); ys, ss = yv[order], s[order]
    keep = np.r_[np.flatnonzero(np.diff(ss)), len(ss) - 1]                    # one point per distinct score
    tp, fp = np.cumsum(ys)[keep], np.cumsum(1 - ys)[keep]
    return np.r_[0, fp / fp[-1]], np.r_[0, tp / tp[-1]]
ROC = {}
for key, fn in SCORE.items():
    tprs = []
    for d in BY.values():
        f, t = roc_xy(d.y.to_numpy(), fn(d))
        tprs.append(np.interp(GRID, f + np.arange(len(f)) * 1e-9, t))
    ROC[key] = np.sum(W[:, None] * np.array(tprs), axis=0) / W.sum()

# label shuffles within cohorts, for the pooled AUCs
rng = np.random.default_rng(42)
PERM = {}
for key in ("core", "panel"):
    obs = pooled(lambda d: fast_auc(d.y.to_numpy(), SCORE[key](d)))
    null = np.array([pooled(lambda d: fast_auc(rng.permutation(d.y.to_numpy()), SCORE[key](d))) for _ in range(10000)])
    PERM[key] = (np.sum(null >= obs) + 1) / (len(null) + 1)

# neuron content: leave-one-out logistic models inside each cohort, then pooled like the AUCs
def loo(F, yv):
    return fast_auc(yv, cross_val_predict(LogisticRegression(), F, yv, cv=LeaveOneOut(), method="predict_proba")[:, 1])
LOO = {"neurons": pooled(lambda d: loo(d[["neuron_score"]].to_numpy(), d.y.to_numpy())),
       "neurons+core": pooled(lambda d: loo(d[["neuron_score", "frozen"]].to_numpy(), d.y.to_numpy())),
       "neurons+panel": pooled(lambda d: loo(d[["neuron_score", "panel"]].to_numpy(), d.y.to_numpy()))}
P["core_centred"] = P.frozen - P.groupby("cohort").frozen.transform("mean")
P["panel_centred"] = P.panel - P.groupby("cohort").panel.transform("mean")
RHO = {"core": spearmanr(P.neuron_score, P.core_centred)[0], "panel": spearmanr(P.neuron_score, P.panel_centred)[0]}

A = POOL["auc"]
NEURO_TABLE = [("neuron markers alone", A["neuron"]["auc"], False),
               ("core classifier", A["frozen"]["auc"], False),
               ("core, neuron content removed", A["frozen|neuron_removed"]["auc"], True),
               ("panel forest", A["panel"]["auc"], False),
               ("panel forest, neuron content removed", A["panel|neuron_removed"]["auc"], True),
               ("neurons alone, leave-one-out", LOO["neurons"], False),
               ("neurons + core score, leave-one-out", LOO["neurons+core"], False),
               ("neurons + panel score, leave-one-out", LOO["neurons+panel"], False)]
UNI = {"people": POOL["people"], "control": POOL["control"], "PD": POOL["PD"], "cohorts": COHORTS,
       "auc": A, "calls@0.5": POOL["calls@0.5"], "calls@oob": POOL["calls@oob"], "perm_p_within_cohorts": PERM,
       "loo": LOO, "spearman_with_neuron_centred": RHO, "gene_agreement": SUM["gene_agreement"],
       "sensitivity_without_GSE7621": {k: SUM["pooled"]["unseen"]["auc"][k] for k in ("frozen", "panel", "neuron")}}
json.dump(UNI, open(OUT / "external_unified_summary.json", "w"), indent=1, default=float)
print(f"{len(COHORTS)} cohorts, {POOL['people']} people ({POOL['control']} control, {POOL['PD']} PD)")
for lab, v, _ in NEURO_TABLE:
    print(f"  {lab:40s} {v:.3f}")
print("  label shuffles within cohorts:", {k: f"{v:.5f}" for k, v in PERM.items()}, " rho with neuron content:", {k: round(v, 2) for k, v in RHO.items()})
print("  area under the averaged curves:", {k: round(float(np.trapz(v, GRID)), 3) for k, v in ROC.items()})


In [ ]:
import matplotlib
try:
    get_ipython(); IN_NB = True
except NameError:
    IN_NB = False; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Rectangle, Polygon, FancyBboxPatch
from matplotlib.path import Path as MPath
from matplotlib.patches import PathPatch

INK, MUTED, RULE = "#1B1D20", "#5E656D", "#C9CED4"
PD_C, CT_C = "#7A2533", "#6F829A"
CORE_C, PANEL_C, NEURO_C = "#23324A", "#8E1B2E", "#9AA1A9"
SETS = {"both": ("#8E1B2E", "#5E0F1C", "Boruta & DEG"), "boruta": ("#2E5A87", "#1B3A5C", "Boruta only")}
EFFECT = LinearSegmentedColormap.from_list("effect", ["#1E3350", "#4C6583", "#9AAABB", "#EEECE7", "#C3A09E", "#8C4B52", "#551C28"])
FONT_STACK = ["Helvetica Neue", "Helvetica", "Arial", "Liberation Sans", "Nimbus Sans", "FreeSans", "DejaVu Sans"]
plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": FONT_STACK, "font.size": 6.3,
                     "axes.linewidth": 0.5, "axes.edgecolor": "#30343A", "axes.labelcolor": INK, "text.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False, "xtick.labelsize": 5.9, "ytick.labelsize": 5.9,
                     "xtick.major.width": 0.5, "ytick.major.width": 0.5, "xtick.major.size": 2.1, "ytick.major.size": 2.1,
                     "xtick.major.pad": 1.7, "ytick.major.pad": 1.7, "xtick.color": "#30343A", "ytick.color": "#30343A",
                     "pdf.fonttype": 42, "ps.fonttype": 42, "figure.dpi": 150, "savefig.facecolor": "white",
                     "figure.facecolor": "white", "mathtext.fontset": "custom", "mathtext.rm": "sans", "mathtext.it": "sans:italic"})

class Canvas:
    """A print-size figure drawn on a millimetre grid."""
    def __init__(self, w, h):
        self.W, self.H = w, h
        self.fig = plt.figure(figsize=(w / 25.4, h / 25.4))
        self.M = self.fig.add_axes([0, 0, 1, 1]); self.M.set_xlim(0, w); self.M.set_ylim(0, h); self.M.axis("off")
    def ax(self, x, y, w, h):
        return self.fig.add_axes([x / self.W, y / self.H, w / self.W, h / self.H])
    def letter(self, x, y, L, title):
        self.M.text(x, y, L, ha="left", va="baseline", fontsize=9, fontweight="bold", color=INK)
        self.M.text(x + 4.2, y, title, ha="left", va="baseline", fontsize=7.2, color=INK)
    def rule(self, x0, x1, y, lw=0.4, color=RULE):
        self.M.plot([x0, x1], [y, y], color=color, lw=lw, solid_capstyle="butt")
    def text(self, x, y, s, **kw):
        kw.setdefault("va", "center"); kw.setdefault("fontsize", 6.0)
        return self.M.text(x, y, s, **kw)
    def save(self, stem):
        for ext in ("pdf", "png"):
            self.fig.savefig(OUT / f"{stem}.{ext}", dpi=600 if ext == "png" else None)
        print("saved", stem)
        plt.show() if IN_NB else plt.close(self.fig)
f2 = lambda v: f"{v:.2f}"
def pfmt(p): return f"= {p:.3f}" if p >= 0.001 else "< 0.001"   # "P = 0.003", "P < 0.0001"


## 2. Figure 6 - the frozen models across the eight cohorts (183 x 155 mm)

In [ ]:
C = Canvas(183.0, 155.0); M = C.M
n_c, n_p = POOL["control"], POOL["PD"]

# ---------------- a: the frozen models, averaged over the eight cohorts ----------------
C.letter(2.0, 151.0, "a", "Frozen models in eight independent bulk cohorts")
axA = C.ax(12.0, 99.0, 46.0, 46.0)
axA.plot([0, 1], [0, 1], color="#B7BDC4", lw=0.5, ls=(0, (2, 2)))
for key, col, ls, lw in (("neuron", NEURO_C, (0, (1.2, 1.2)), 1.0), ("panel", PANEL_C, (0, (3, 1.6)), 0.9), ("core", CORE_C, "-", 1.3)):
    axA.plot(GRID, ROC[key], color=col, ls=ls, lw=lw, solid_joinstyle="miter")
axA.set_xlim(-0.01, 1.01); axA.set_ylim(-0.01, 1.01); axA.set_aspect("equal")
axA.set_xticks([0, 0.5, 1]); axA.set_yticks([0, 0.5, 1]); axA.set_xticklabels(["0", "0.5", "1"]); axA.set_yticklabels(["0", "0.5", "1"])
axA.set_xlabel("False-positive rate", fontsize=6.5, labelpad=2); axA.set_ylabel("True-positive rate", fontsize=6.5, labelpad=2)
KEY = [("core classifier", A["frozen"]["auc"], CORE_C, "-", 1.3, True), ("Boruta-panel forest", A["panel"]["auc"], PANEL_C, (0, (3, 1.6)), 0.9, False),
       ("neuron markers alone", A["neuron"]["auc"], NEURO_C, (0, (1.2, 1.2)), 1.0, False)]
for i, (lab, a, col, ls, lw, b) in enumerate(KEY):
    yy = 0.30 - i * 0.095
    axA.plot([0.22, 0.31], [yy, yy], color=col, ls=ls, lw=lw, transform=axA.transAxes)
    axA.text(0.34, yy, lab, transform=axA.transAxes, ha="left", va="center", fontsize=5.9, color=INK, fontweight="bold" if b else "normal")
    axA.text(1.0, yy, f2(a), transform=axA.transAxes, ha="right", va="center", fontsize=5.9, color=INK, fontweight="bold" if b else "normal")
axA.text(1.0, 0.30 + 0.095, "pooled AUC", transform=axA.transAxes, ha="right", va="center", fontsize=5.5, color=MUTED)
cA = A["frozen"]
C.text(12.0, 87.6, f"{len(COHORTS)} cohorts, {POOL['people']} people ({n_c} control, {n_p} PD); trained only on the 63 laser-capture people",
       fontsize=5.7, color=MUTED)
C.text(12.0, 84.6, f"core classifier AUC {cA['auc']:.2f} (95% CI {cA['ci_lo']:.2f}–{cA['ci_hi']:.2f}), label shuffles within cohorts "
       f"P {pfmt(PERM['core'])}", fontsize=5.7, color=MUTED)
C.text(12.0, 81.6, "curves: each cohort's ROC, averaged with weights equal to its PD–control pairs", fontsize=5.7, color=MUTED)

# ---------------- b: how much of it is neuron loss ----------------
C.letter(72.0, 151.0, "b", "How much of it is neuron loss")
axB = C.ax(82.0, 99.0, 46.0, 46.0)
for lab, col in ((0, CT_C), (1, PD_C)):
    m = P.y.to_numpy() == lab
    axB.scatter(P.neuron_score[m], P.core_centred[m], s=8, color=col, edgecolor="white", linewidth=0.3, zorder=3)
xx = np.linspace(P.neuron_score.min(), P.neuron_score.max(), 50)
axB.plot(xx, np.polyval(np.polyfit(P.neuron_score, P.core_centred, 1), xx), color="#8C9299", lw=0.7, ls=(0, (3, 1.6)), zorder=2)
axB.axhline(0, color="#D5D9DE", lw=0.4, zorder=1)
axB.set_xlabel("dopamine-neuron content (8 marker genes, z)", fontsize=6.3, labelpad=2)
axB.set_ylabel("core classifier score, centred within cohort", fontsize=6.3, labelpad=2)
axB.text(0.98, 0.97, f"Spearman ρ = {RHO['core']:.2f}", transform=axB.transAxes, ha="right", va="top", fontsize=5.8, color=MUTED)
for x, col, t in ((84.0, CT_C, "control"), (97.0, PD_C, "PD")):
    M.scatter([x], [146.2], s=13, color=col, edgecolor="white", linewidth=0.4)
    C.text(x + 1.6, 146.2, t, fontsize=5.8)
TX0, TX1, TY = 139.0, 181.0, 141.5
C.rule(TX0, TX1, TY + 2.2, lw=0.6, color="#3A3F45")
C.text(TX1, TY, "pooled AUC", ha="right", fontsize=5.9, color=MUTED)
C.rule(TX0, TX1, TY - 1.8, lw=0.35, color="#8C9299")
for i, (lab, v, bold) in enumerate(NEURO_TABLE):
    yy = TY - 4.5 - i * 4.0
    C.text(TX0, yy, lab, ha="left", fontsize=5.9, color=INK, fontweight="bold" if bold else "normal")
    C.text(TX1, yy, f2(v), ha="right", fontsize=5.9, color=INK, fontweight="bold" if bold else "normal")
C.rule(TX0, TX1, TY - 4.5 - (len(NEURO_TABLE) - 1) * 4.0 - 2.4, lw=0.6, color="#3A3F45")
C.text(TX0, TY - 4.5 - (len(NEURO_TABLE) - 1) * 4.0 - 5.4, "neuron content removed: score regressed on the", fontsize=5.4, color=MUTED, ha="left")
C.text(TX0, TY - 4.5 - (len(NEURO_TABLE) - 1) * 4.0 - 8.0, "neuron score within each cohort", fontsize=5.4, color=MUTED, ha="left")

# ---------------- c: every Boruta gene, cohort by cohort ----------------
C.rule(2.0, 181.0, 79.0)
C.letter(2.0, 74.5, "c", "Every Boruta gene: effect in the laser-capture cohort and in each bulk cohort")
G = GMT.copy(); G["grp"] = np.where(G.deg, 0, 1)
G = G.sort_values(["grp", "g_lcm"], ascending=[True, False]).reset_index(drop=True)
nG = len(G); X0, X1 = 44.0, 164.0; cw = (X1 - X0) / nG
ch, st = 3.4, 3.8
ROWS = [("laser-capture, pooled", 64.0, dict(zip(G.gene, G.g_lcm)))]
y_ = 64.0 - st - 1.4
EFFC = GBC.set_index(["cohort", "gene"])
for c in COHORTS:
    ROWS.append((c, y_, {g: EFFC.loc[(c, g), "g_adj"] for g in G.gene if (c, g) in EFFC.index})); y_ -= st
y_ -= 1.4
ROWS.append((f"{len(COHORTS)} cohorts pooled", y_, dict(zip(G.gene, G.g_external)))); y_ -= st
ROWS.append(("pooled, neuron content removed", y_, dict(zip(G.gene, G.g_external_neuron_adj))))
GMX = 1.5; norm = Normalize(-GMX, GMX)
lcm = dict(zip(G.gene, G.g_lcm))
for r_i, (lab, yy, vals) in enumerate(ROWS):
    bold = r_i == len(ROWS) - 1
    C.text(X0 - 1.5, yy, lab, ha="right", fontsize=5.9, color=INK, fontweight="bold" if bold else "normal")
    k = n = 0
    for j, g in enumerate(G.gene):
        v = vals.get(g, np.nan)
        face = "#E6E8EB" if not np.isfinite(v) else EFFECT(norm(np.clip(v, -GMX, GMX)))
        M.add_patch(Rectangle((X0 + j * cw, yy - ch / 2), cw, ch, facecolor=face, edgecolor="white", lw=0.6))
        if r_i and np.isfinite(v):
            n += 1; k += int(np.sign(v) == np.sign(lcm[g]))
            if np.sign(v) != np.sign(lcm[g]):
                M.plot([X0 + j * cw + 0.8, X0 + (j + 1) * cw - 0.8], [yy - ch / 2 + 0.6, yy + ch / 2 - 0.6], color="#8C9299", lw=0.4)
    if r_i:
        C.text(181.0, yy, f"{k}/{n}", ha="right", fontsize=6.0, color=INK, fontweight="bold" if bold else "normal")
C.text(181.0, ROWS[0][1], "same sign", ha="right", fontsize=5.7, color=MUTED)
cy0, cy1 = ROWS[1][1] + ch / 2, ROWS[len(COHORTS)][1] - ch / 2
M.plot([30.0, 30.0], [cy1, cy0], color="#9AA1A9", lw=0.5)
for yv in (cy0, cy1):
    M.plot([30.0, 31.0], [yv, yv], color="#9AA1A9", lw=0.5)
C.text(28.2, (cy0 + cy1) / 2, "neuron content removed", ha="center", rotation=90, fontsize=5.6, color=MUTED)
ylab = ROWS[-1][1] - ch / 2 - 0.8
for j, r in G.iterrows():
    C.text(X0 + (j + 0.5) * cw, ylab, r.symbol, ha="center", va="top", rotation=90, fontsize=5.7, fontstyle="italic",
           color=SETS["both" if r.deg else "boruta"][1])
for grp, key in ((0, "both"), (1, "boruta")):
    js = np.where(G.grp.to_numpy() == grp)[0]
    if len(js):
        M.add_patch(Rectangle((X0 + js.min() * cw + 0.2, ROWS[0][1] + ch / 2 + 1.0), (js.max() - js.min() + 1) * cw - 0.4, 0.8,
                              facecolor=SETS[key][0], edgecolor="none"))
        C.text(X0 + (js.min() + js.max() + 1) / 2 * cw, ROWS[0][1] + ch / 2 + 2.4, f"{SETS[key][2]}  ({len(js)})", ha="center",
               va="bottom", color=SETS[key][1])
cax = C.ax(8.0, 2.2, 22.0, 1.6)
cax.imshow(np.linspace(0, 1, 256)[None, :], aspect="auto", cmap=EFFECT); cax.set_xticks([]); cax.set_yticks([])
for s_ in cax.spines.values():
    s_.set_linewidth(0.3); s_.set_color("#B7BDC4")
C.text(7.0, 3.0, f"−{GMX:g}", ha="right", fontsize=5.7, color=MUTED)
C.text(31.0, 3.0, f"+{GMX:g}   Hedges' g, PD minus control", ha="left", fontsize=5.7, color=MUTED)
ga = SUM["gene_agreement"]["g_external_neuron_adj"]
C.text(181.0, 3.0, f"struck through: opposite sign to discovery  ·  grey: not measured  ·  bottom row: binomial P {pfmt(ga['binom_p'])}, "
       f"Spearman ρ = {ga['spearman']:.2f}", ha="right", fontsize=5.6, color=MUTED)
C.save("Figure06_external_validation_unified")


## 3. Figure 7 - cohort by cohort, calls, donor overlap (183 x 150 mm)

In [ ]:
C = Canvas(183.0, 150.0); M = C.M
PLAT = {"GSE7621": "array (U133 Plus 2)", "GSE8397": "array (U133A)", "GSE20163": "array (U133A)", "GSE20164": "array (U133A)",
        "GSE20292": "array (U133A)", "GSE49036": "array (U133 Plus 2)", "GSE114517": "RNA-seq", "GSE168496": "RNA-seq"}
def mark(x, y_, color, filled=True, size=11):
    M.scatter([x], [y_], s=size, facecolor=color if filled else "white", edgecolor=color, linewidth=0.8, zorder=5, clip_on=False)

# ======================= a: forest plot, every cohort alike =======================
C.letter(2.0, 145.0, "a", "The frozen models, cohort by cohort")
Y0, STEP = 131.0, 4.9
ROWY = {c: Y0 - i * STEP for i, c in enumerate(COHORTS)}
PYY = ROWY[COHORTS[-1]] - 7.0
YB, YT = PYY - 3.2, Y0 + 3.2
AX = {"core": (76.0, 34.0), "panel": (127.0, 33.0)}
NUMX = {"core": (114.0, 120.0), "panel": (164.3, 170.2)}
XNEU = 177.5
HY = YT + 2.4
for x, t, ha in ((4.0, "cohort", "left"), (21.0, "platform", "left"), (52.5, "control / PD", "center"), (65.5, "shared donors\nremoved", "center")):
    C.text(x, HY, t, ha=ha, va="bottom", fontsize=5.9, color=MUTED, linespacing=1.05)
for key, (x0, w) in AX.items():
    col = CORE_C if key == "core" else PANEL_C
    C.text(x0 + w / 2, HY + 3.4, "core classifier" if key == "core" else "Boruta-panel forest", ha="center", va="bottom",
           fontsize=6.6, color=col, fontweight="bold")
    C.text(x0 + w / 2, HY, "AUC", ha="center", va="bottom", fontsize=5.9, color=MUTED)
    mark(NUMX[key][0], HY + 1.0, col, True, 9); mark(NUMX[key][1], HY + 1.0, col, False, 9)
M.scatter([XNEU], [HY + 1.0], s=16, marker="|", color=NEURO_C, linewidth=1.1, clip_on=False)
C.text(XNEU, HY + 3.0, "neuron\nmarkers", ha="center", va="bottom", fontsize=5.6, color=MUTED, linespacing=1.0)
C.rule(2.0, 181.0, YT + 0.9, lw=0.6, color="#30343A")
AXES = {}
for key, (x0, w) in AX.items():
    ax = C.ax(x0, YB, w, YT - YB); ax.set_xlim(0.12, 1.02); ax.set_ylim(YB, YT)
    ax.spines["left"].set_visible(False); ax.set_yticks([])
    ax.set_xticks([0.25, 0.5, 0.75, 1.0]); ax.set_xticklabels(["0.25", "0.5", "0.75", "1"])
    ax.axvline(0.5, color="#B7BDC4", lw=0.5, ls=(0, (2, 2)), zorder=0)
    ax.patch.set_alpha(0); AXES[key] = ax
    C.text(x0 + w / 2, YB - 5.4, "AUC in the external cohort", ha="center", va="top")
for i, c in enumerate(COHORTS):
    if i % 2 == 0:
        M.add_patch(Rectangle((2.0, ROWY[c] - STEP / 2), 179.0, STEP, color="#F5F4F1", lw=0, zorder=0))
for c in COHORTS:
    yy = ROWY[c]; r = RES.loc[(c, "frozen")]
    C.text(4.0, yy, c, ha="left", fontsize=6.2)
    C.text(21.0, yy, PLAT[c], ha="left", fontsize=5.9, color=MUTED)
    C.text(52.5, yy, f"{int(r.control)} / {int(r.PD)}", ha="center")
    k = int(OVL.loc[c, "shared_with_discovery"])
    C.text(65.5, yy, str(k) if k else "–", ha="center", color=INK if k else MUTED, fontweight="bold" if k else "normal")
    for key, (m_rep, m_str) in (("core", ("frozen", "strict")), ("panel", ("panel", "panel_strict"))):
        ax = AXES[key]; col = CORE_C if key == "core" else PANEL_C
        a, s_ = RES.loc[(c, m_rep)], RES.loc[(c, m_str)]
        ax.plot([a.ci_lo, a.ci_hi], [yy, yy], color=col, lw=0.8, solid_capstyle="butt", zorder=2)
        ax.scatter([a.auc_neuron_markers], [yy], s=22, marker="|", color=NEURO_C, linewidth=1.1, zorder=3)
        ax.scatter([s_.auc], [yy], s=13, facecolor="white", edgecolor=col, linewidth=0.8, zorder=4)
        ax.scatter([a.auc], [yy], s=13, color=col, zorder=5, linewidth=0)
        C.text(NUMX[key][0], yy, f2(a.auc), ha="center")
        C.text(NUMX[key][1], yy, f2(s_.auc), ha="center", color=MUTED)
    C.text(XNEU, yy, f2(r.auc_neuron_markers), ha="center", color=MUTED)
C.rule(2.0, 181.0, PYY + STEP / 2 + 0.6, lw=0.5, color="#30343A")
C.text(4.0, PYY, f"pooled, {len(COHORTS)} cohorts", ha="left", fontsize=6.2, fontweight="bold")
C.text(52.5, PYY, f"{POOL['control']} / {POOL['PD']}", ha="center", fontweight="bold")
C.text(65.5, PYY, str(int(OVL.shared_with_discovery.sum())), ha="center", fontweight="bold")
for key, (m_rep, m_str) in (("core", ("frozen", "strict")), ("panel", ("panel", "panel_strict"))):
    ax = AXES[key]; col = CORE_C if key == "core" else PANEL_C
    a, s_ = A[m_rep], A[m_str]; h = 1.45
    ax.add_patch(Polygon([[a["ci_lo"], PYY], [a["auc"], PYY + h], [a["ci_hi"], PYY], [a["auc"], PYY - h]], closed=True,
                         facecolor=col, edgecolor="none", zorder=4))
    ax.scatter([s_["auc"]], [PYY], s=13, facecolor="white", edgecolor=col, linewidth=0.8, zorder=5)
    ax.scatter([A["neuron"]["auc"]], [PYY], s=22, marker="|", color=NEURO_C, linewidth=1.1, zorder=3)
    C.text(NUMX[key][0], PYY, f2(a["auc"]), ha="center", fontweight="bold")
    C.text(NUMX[key][1], PYY, f2(s_["auc"]), ha="center", color=MUTED)
C.text(XNEU, PYY, f2(A["neuron"]["auc"]), ha="center", color=MUTED)
KY = YB - 11.0
mark(4.8, KY, "#4A4F56", True, 11); C.text(6.8, KY, "model as reported (trained on all 63 discovery people), with 95% CI", ha="left", fontsize=5.9)
mark(84.8, KY, "#4A4F56", False, 11)
C.text(86.8, KY, "strictly independent: same recipe, retrained without the discovery studies of the same source (c)", ha="left", fontsize=5.9)
M.scatter([4.8], [KY - 3.6], s=22, marker="|", color=NEURO_C, linewidth=1.1)
C.text(6.8, KY - 3.6, "8 dopamine-neuron marker genes alone (fewer neurons = more PD-like)", ha="left", fontsize=5.9)
C.text(84.0, KY - 3.6, "pooled: cohort AUCs weighted by PD–control pairs, each donor once; diamond width = 95% CI", ha="left",
       fontsize=5.6, color=MUTED)
TOP = KY - 7.0
C.rule(2.0, 181.0, TOP, lw=0.4)

# ======================= b: calls at cut-offs fixed before testing =======================
C.letter(2.0, TOP - 4.6, "b", "Calls at cut-offs fixed before testing")
T0 = TOP - 8.4
C.rule(4.0, 123.0, T0, lw=0.6, color="#30343A")
G1, G2 = (61.0, 71.5, 82.0), (96.0, 106.5, 117.0)
for xs, lab in ((G1, "cut-off 0.5"), (G2, "discovery out-of-bag cut-off")):
    C.text(np.mean(xs), T0 - 2.3, lab, ha="center", fontsize=5.8, color=INK)
    C.rule(xs[0] - 5.0, xs[-1] + 5.0, T0 - 4.1, lw=0.35, color="#8C9299")
    for x, t in zip(xs, ("accuracy", "sensitivity", "specificity")):
        C.text(x, T0 - 6.0, t, ha="center", fontsize=5.5, color=MUTED)
C.text(44.0, T0 - 6.0, "AUC (95% CI)", ha="center", fontsize=5.5, color=MUTED)
C.rule(4.0, 123.0, T0 - 8.0, lw=0.35, color="#8C9299")
TROWS = [("frozen", "core classifier", CORE_C, True), ("strict", "strictly independent", CORE_C, False),
         ("panel", "Boruta-panel forest", PANEL_C, True), ("panel_strict", "strictly independent", PANEL_C, False)]
for i, (key, lab, col, filled) in enumerate(TROWS):
    yy = T0 - 11.0 - i * 4.2
    mark(6.0, yy, col, filled, 10)
    C.text(8.5, yy, lab, ha="left", fontsize=6.0, color=INK if filled else MUTED, fontweight="bold" if key == "panel" else "normal")
    a = A[key]
    C.text(44.0, yy, f"{a['auc']:.2f} ({a['ci_lo']:.2f}–{a['ci_hi']:.2f})", ha="center", fontweight="bold" if key == "panel" else "normal")
    for xs, calls in ((G1, POOL["calls@0.5"][key]), (G2, POOL["calls@oob"][key])):
        for x, m in zip(xs, ("accuracy", "sensitivity", "specificity")):
            C.text(x, yy, f2(calls[m]), ha="center")
TB = T0 - 11.0 - 3 * 4.2 - 2.6
C.rule(4.0, 123.0, TB, lw=0.6, color="#30343A")
TH = SUM["thresholds"]
C.text(4.0, TB - 2.8, f"Pooled over {len(COHORTS)} cohorts ({POOL['people']} people). Out-of-bag cut-offs set on the 63 discovery people: "
       f"core {TH['core']:.2f}, panel {TH['panel']:.2f}.", ha="left", fontsize=5.4, color=MUTED)
# balanced accuracy, cohort by cohort
axS = C.ax(16.0, 6.6, 106.0, TB - 9.6 - 6.6)
xs = np.arange(len(COHORTS))
for key, col, dx in (("frozen", CORE_C, -0.12), ("panel", PANEL_C, 0.12)):
    v = [RES.loc[(c, key), "balanced_accuracy@oob"] for c in COHORTS]
    axS.scatter(xs + dx, v, s=11, color=col, zorder=3, linewidth=0)
axS.axhline(0.5, color="#B7BDC4", lw=0.5, ls=(0, (2, 2)), zorder=0)
axS.set_xlim(-0.5, len(COHORTS) - 0.5); axS.set_ylim(0.2, 1.02)
axS.set_yticks([0.25, 0.5, 0.75, 1]); axS.set_yticklabels(["0.25", "0.5", "0.75", "1"])
axS.set_xticks(xs); axS.set_xticklabels(COHORTS, fontsize=5.5)
axS.tick_params(axis="x", length=0, pad=2.0)
axS.set_ylabel("balanced\naccuracy", fontsize=5.7, labelpad=2, linespacing=1.0)
axS.patch.set_alpha(0)
for c_i in range(len(COHORTS)):
    if c_i % 2 == 0:
        axS.axvspan(c_i - 0.5, c_i + 0.5, color="#F5F4F1", lw=0, zorder=-1)
C.text(16.0, TB - 6.4, "each cohort, balanced accuracy at the out-of-bag cut-off:", ha="left", fontsize=5.4, color=MUTED)
for x, col, t in ((72.0, CORE_C, "core classifier"), (89.0, PANEL_C, "Boruta-panel forest")):
    M.scatter([x], [TB - 6.4], s=9, color=col); C.text(x + 1.3, TB - 6.4, t, ha="left", fontsize=5.4, color=MUTED)

# ======================= c: which brains overlap =======================
XC = 128.0
C.letter(XC, TOP - 4.6, "c", "Shared brains, found and removed")
LX0, LX1, RX0, RX1 = 129.5, 146.5, 157.0, 179.5
BH = 3.4
C.text((LX0 + LX1) / 2, TOP - 9.2, "discovery (laser capture)", ha="center", fontsize=5.6, color=MUTED)
C.text((RX0 + RX1) / 2, TOP - 9.2, "external (bulk)", ha="center", fontsize=5.6, color=MUTED)
FAM = [("one laboratory series", ["GSE20141", "GSE24378"], ["GSE20292", "GSE20163", "GSE20164"]),
       ("Netherlands Brain Bank", ["GSE182622"], ["GSE168496", "GSE49036"]),
       ("other brain banks", ["GSE169755"], ["GSE7621", "GSE8397", "GSE114517"])]
EXT_N = {c: int(OVL.loc[c, "people"]) for c in COHORTS}
KEPT = {c: int(RES.loc[(c, "frozen"), "n"]) for c in COHORTS}
yy = TOP - 12.2
POS = {}
for f_i, (fam, left, right) in enumerate(FAM):
    top = yy
    yy -= 2.8
    rys = []
    for c in right:
        POS[c] = (yy - BH / 2); rys.append(yy - BH / 2); yy -= BH + 0.9
    bottom = yy + 0.9 - 0.8
    if f_i < 2:
        M.add_patch(Rectangle((XC + 0.5, bottom), 181.0 - XC - 0.5, top - bottom, color="#F3F1EC", lw=0, zorder=0))
    C.text(XC + 1.5, top - 1.3, fam, ha="left", fontsize=5.5, color=MUTED, fontstyle="italic")
    if len(left) == 1:
        POS[left[0]] = rys[0] if fam != "other brain banks" else np.mean(rys)
    else:
        POS[left[0]] = np.mean(rys[:2]); POS[left[1]] = rys[2]
    yy -= 1.2
def box(x0, x1, yc, name, right_txt, strong=False):
    M.add_patch(FancyBboxPatch((x0, yc - BH / 2), x1 - x0, BH, boxstyle="round,pad=0,rounding_size=0.6",
                               facecolor="white", edgecolor="#30343A" if strong else "#9AA1A9", lw=0.5, zorder=3))
    C.text(x0 + 1.0, yc, name, ha="left", fontsize=5.5, zorder=4)
    C.text(x1 - 1.0, yc, right_txt, ha="right", fontsize=5.4, color=MUTED, zorder=4)
for c, n in DISC_N.items():
    box(LX0, LX1, POS[c], c, str(n))
for c in COHORTS:
    box(RX0, RX1, POS[c], c, f"{EXT_N[c]}$\\rightarrow${KEPT[c]}" if KEPT[c] != EXT_N[c] else str(EXT_N[c]), strong=KEPT[c] != EXT_N[c])
def link(y0, y1, color, ls, label, lw=0.9):
    xa, xb = LX1, RX0; xm = (xa + xb) / 2
    path = MPath([(xa, y0), (xm, y0), (xm, y1), (xb, y1)], [MPath.MOVETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4])
    M.add_patch(PathPatch(path, facecolor="none", edgecolor=color, lw=lw, ls=ls, zorder=2))
    C.text(xm, (y0 + y1) / 2, label, ha="center", fontsize=5.8, color=color, fontweight="bold",
           bbox=dict(boxstyle="round,pad=0.12", facecolor="white", edgecolor="none"), zorder=4)
m = MATCH.groupby("cohort").size().to_dict()
link(POS["GSE20141"], POS["GSE20292"], PD_C, "-", str(m.get("GSE20292", 0)), lw=1.4)
link(POS["GSE20141"], POS["GSE20163"], PD_C, "-", str(m.get("GSE20163", 0)), lw=0.9)
link(POS["GSE182622"], POS["GSE168496"], "#8C9299", (0, (2, 1.5)), "0", lw=0.7)
dup = int(OVL.loc["GSE20163", "also_in_another_external_cohort"])
if dup:
    xb = RX1 + 1.0
    M.plot([RX1, xb, xb, RX1], [POS["GSE20292"], POS["GSE20292"], POS["GSE20163"], POS["GSE20163"]], color=PD_C, lw=0.6)
    C.text(xb + 0.6, (POS["GSE20292"] + POS["GSE20163"]) / 2, str(dup), ha="left", fontsize=5.6, color=PD_C, fontweight="bold")
LG = 3.2
M.plot([XC + 1.0, XC + 5.0], [LG + 2.8, LG + 2.8], color=PD_C, lw=1.1)
C.text(XC + 6.0, LG + 2.8, "same donor IDs: removed before testing", ha="left", fontsize=5.3, color=MUTED)
M.plot([XC + 1.0, XC + 5.0], [LG, LG], color="#8C9299", lw=0.7, ls=(0, (2, 1.5)))
C.text(XC + 6.0, LG, "IDs compared, none shared   ·   people before$\\rightarrow$after", ha="left", fontsize=5.3, color=MUTED)
C.save("Figure07_external_multicohort_unified")


## 4. Ready-to-paste legends, methods and numbers

In [ ]:
cF, cP, cN = A["frozen"], A["panel"], A["neuron"]
ga, gr = SUM["gene_agreement"]["g_external_neuron_adj"], SUM["gene_agreement"]["g_external"]
co5, cob = POOL["calls@0.5"], POOL["calls@oob"]
nshared = int(OVL.shared_with_discovery.sum())
sw = UNI["sensitivity_without_GSE7621"]
ci = lambda d: f"{d['auc']:.2f} (95% CI {d['ci_lo']:.2f}-{d['ci_hi']:.2f})"
TEXT = f"""FIGURE 6 LEGEND - External validation in eight independent bulk substantia nigra cohorts
Both models were frozen before any external cohort was analysed and were applied without refitting; each cohort's genes were
z-scored within the cohort and ranked within each person, as in discovery. {len(COHORTS)} cohorts, {POOL['people']} people ({POOL['control']} control, {POOL['PD']} PD),
after removal of {nshared} donors shared with the discovery data. (a) ROC of the core classifier, the Boruta-panel forest and an eight-gene
dopamine-neuron marker score, each cohort's curve averaged with weights equal to its number of PD-control pairs; pooled AUCs
{cF['auc']:.2f}, {cP['auc']:.2f} and {cN['auc']:.2f} (label shuffles within cohorts: P {pfmt(PERM['core'])} and P {pfmt(PERM['panel'])} for the two models). (b) Core
classifier score (centred within each cohort) against dopamine-neuron content (Spearman {RHO['core']:.2f}). Table: pooled AUCs with the
neuron-content component regressed out of each score within each cohort, and leave-one-out logistic models within each cohort
using neuron content alone or with a model score. (c) PD-versus-control effect (Hedges' g) of each Boruta gene in the laser-capture
discovery cohort, in each bulk cohort after neuron content is regressed out, and pooled across the bulk cohorts by inverse variance
without and with this adjustment; struck-through cells have the opposite sign to discovery, grey cells were not measured.

FIGURE 7 LEGEND - The frozen models cohort by cohort, calls at fixed cut-offs, and donor overlap
(a) AUC in each cohort (filled, model as reported, with 95% bootstrap CI; open, strictly independent version of the same recipe
retrained without the discovery studies from the same source; grey tick, neuron markers alone) and pooled over the {len(COHORTS)} cohorts
(cohort AUCs weighted by PD-control pairs, each donor counted once; diamond width, 95% CI). (b) Accuracy, sensitivity and
specificity at two cut-offs fixed on the 63 discovery people before testing: 0.5, and each forest's out-of-bag accuracy optimum;
below, balanced accuracy at the out-of-bag cut-off in each cohort. (c) Donor overlap between the discovery studies and the bulk
cohorts. Donor IDs were compared where they were public in a comparable format; shared donors ({nshared}) were removed before testing,
and one donor present in two bulk cohorts was counted once when cohorts were pooled. Shaded groups may share a brain source, so
their cohorts were also scored by the strictly independent models.

METHODS - External validation
Model lock. The core classifier (within-person gene ranks, 30 principal components, Random Forest of 1,000 trees) was chosen by a
sweep over 97 Random Forest variants that used only the 63 discovery people; the Boruta panel and its forest were fitted on the
same people. Both were frozen, with their decision thresholds, before the external cohorts were analysed.
Cohorts. Eight public bulk substantia nigra cohorts were obtained from GEO and analysed together: GSE7621, GSE8397, GSE20163,
GSE20164 and GSE20292 (Affymetrix HG-U133A or Plus 2; for GSE8397, lateral and medial nigra averaged per case), GSE49036
(HG-U133 Plus 2; controls and PD, Braak 3-6), GSE114517 (RNA-seq counts, nigra only; PD with dementia) and GSE168496 (RNA-seq,
transcript abundances summed to genes). Probe sets were mapped to Ensembl genes with g:Profiler (highest-mean probe set per gene)
and array data were log2-transformed. Each cohort was processed without its labels: genes z-scored within the cohort, then ranked
within each person; unmeasured genes were set to the cohort mean.
Donor overlap. Donor IDs, where public in a comparable format, were matched to the discovery donors (GSE20292 and GSE20163
against GSE20141; GSE168496 against GSE182622); the {nshared} matched donors (all with the same diagnosis) were removed. Because not
every cohort reports IDs, each cohort was also scored by strictly independent models - the same recipe, including Boruta for the
panel, re-run without every discovery study from a potentially shared source (GSE20141 and GSE24378 for GSE20163, GSE20164 and
GSE20292; GSE182622 for GSE49036 and GSE168496).
Statistics. Per-cohort AUCs have 95% CIs from 4,000 bootstrap resamples of people. Pooled AUCs are cohort AUCs weighted by the
number of PD-control pairs (CIs from resampling people within cohorts; P from 10,000 label shuffles within cohorts); ROC curves
were averaged vertically with the same weights. Calls used thresholds fixed on the discovery people: 0.5 and each forest's
out-of-bag accuracy optimum (core {SUM['thresholds']['core']:.2f}, panel {SUM['thresholds']['panel']:.2f}). Neuron content was the mean z-score of eight
dopamine-neuron marker genes (TH, SLC6A3, SLC18A2, DDC, KCNJ6, ALDH1A1, NR4A2, EN1); neuron-adjusted AUCs used each score's residual
after linear regression on it within the cohort; leave-one-out logistic models were fitted within each cohort and pooled like the
AUCs. Gene effects were Hedges' g (PD minus control) with and without neuron content regressed out, pooled by inverse variance;
agreement with discovery was tested with a one-sided binomial test on the signs. GSE7621 had been examined in an earlier version of
this project; without it the pooled AUCs are {sw['frozen']['auc']:.2f} (core), {sw['panel']['auc']:.2f} (panel) and {sw['neuron']['auc']:.2f} (neuron markers).

RESULTS - numbers ({len(COHORTS)} cohorts, {POOL['people']} people)
  core classifier      AUC {ci(cF)}, label shuffles P {pfmt(PERM['core'])}; neuron-adjusted {A['frozen|neuron_removed']['auc']:.2f}; strictly independent {A['strict']['auc']:.2f}
  Boruta-panel forest  AUC {ci(cP)}, label shuffles P {pfmt(PERM['panel'])}; neuron-adjusted {ci(A['panel|neuron_removed'])}
                       strictly independent {ci(A['panel_strict'])}
  neuron markers alone AUC {ci(cN)}
  leave-one-out: neurons {LOO['neurons']:.2f}; neurons + core {LOO['neurons+core']:.2f}; neurons + panel {LOO['neurons+panel']:.2f}
  calls, cut-off 0.5:   core acc {co5['frozen']['accuracy']:.2f} sens {co5['frozen']['sensitivity']:.2f} spec {co5['frozen']['specificity']:.2f}; panel acc {co5['panel']['accuracy']:.2f} sens {co5['panel']['sensitivity']:.2f} spec {co5['panel']['specificity']:.2f}
  calls, OOB cut-off:   core acc {cob['frozen']['accuracy']:.2f} sens {cob['frozen']['sensitivity']:.2f} spec {cob['frozen']['specificity']:.2f}; panel acc {cob['panel']['accuracy']:.2f} sens {cob['panel']['sensitivity']:.2f} spec {cob['panel']['specificity']:.2f}
  genes: {ga['same_sign']}/{ga['n']} same sign after neuron adjustment (binomial P = {ga['binom_p']:.4f}, Spearman {ga['spearman']:.2f}); {gr['same_sign']}/{gr['n']} unadjusted
"""
print(TEXT)
open(OUT / "external_unified_legends_methods.txt", "w").write(TEXT)
